# 2D HPC results | Landscape, structures and PXRD

Open this notebook with the HPC run directory as its working directory. Keep `cof-landscaper_2d.params.json`, the `experimental_pxrd/` folder and the generated COF directory together. Select the Jupyter kernel for your COF-Landscaper environment. [Installation](../../../README.md#installation) · [Jupyter setup](../../../README.md#running-the-notebooks)

In [ ]:
from tempfile import TemporaryDirectory

import coflandscaper as cl

## Read the workflow parameters

The JSON supplies the COF name, mode (`"incl"`, `"serr"` or `"both"`) and initial PXRD display range. To zoom, replace `XLIM` with a pair such as `(3.0, 12.0)` in degrees 2θ. Set `SAVE_COMPARISON_PDFS` to `True` to write comparison PDFs.

In [ ]:
params = cl.utilities.load_params("cof-landscaper_2d.params.json")
COF_NAME = params["COF_NAME"]
MODE = params["MODE"]
XLIM = tuple(params["PXRD_XLIM"])
SAVE_COMPARISON_PDFS = False

## View the stacking energy landscapes

The plot reads the single-point energy tables in `<COF_NAME>/3_<COF_NAME>_landscape/`. It shows relative energy over the sampled interlayer distances and slips. The PNGs created for display go to a temporary directory.

In [ ]:
with TemporaryDirectory() as plot_dir:
    cl.Landscape().run(
        COF_NAME,
        mode=MODE,
        minima_mode=params["MINIMA_MODE"],
        show_minima_markers=params["SHOW_MINIMA_MARKERS"],
        show_title_block=params["SHOW_TITLE_BLOCK"],
        method_label=params["METHOD_LABEL"],
        output_folder=plot_dir,
        show=True,
    )

## Inspect the optimised structures

Rotate and zoom the optimised structures from `<COF_NAME>/4_<COF_NAME>_optimization/`. The viewer reports the final interlayer distance and slip for each CIF.

In [ ]:
cl.VisualizeCOF().visualize_cof(COF_NAME, mode=MODE);

## Compare simulated and experimental PXRD

The plot reads the simulated XY files in `<COF_NAME>/5_<COF_NAME>_analysis/pxrd_xy/` and discovers the single experimental `.xy` file in `experimental_pxrd/`. With PDF saving enabled, figures are written to the mode subfolders of `5_<COF_NAME>_analysis/pxrd_plots/` and can replace matching PDFs.

In [ ]:
cl.PXRD().plot_sim_vs_exp(
    COF_NAME, mode=MODE, xlim=XLIM, show=True, save=SAVE_COMPARISON_PDFS
);

## Optional | Diagnose peak-position differences

After inspecting the comparison, copy this snippet into a code cell and choose 2θ intervals that bracket matching features. The example intervals are starting points for ILCOF-1. The diagnostic reports the intensity-weighted simulated centroid minus the fitted experimental centre. Each matched region contributes equally to the summary. Unmatched regions retain a missing offset. Peak extraction writes or replaces `<structure>_regions.csv` in the mode subfolders of `5_<COF_NAME>_analysis/pxrd_peaks/`.

```python
diagnostics = cl.PXRD()
diagnostics.extract_peak_regions(
    COF_NAME, mode=MODE, peak_regions=[(3.3, 4.3), (7.1, 8.1)]
)
regions, summary = diagnostics.peak_position_diagnostics()
print(regions.to_string(index=False))
summary
```